# 15. Prueba piloto de corrección de intensidad (2023–2024, validación cruzada por tormenta)

Tres bases contra dos referencias, sobre los emparejamientos en t0 de `matches_long.csv` (seguimiento D24):

| Id | Método | Predictores |
|---|---|---|
| raw | AIFS sin corregir (viento / 0.88) | — |
| pers | persistencia: intensidad observada en t0 | — |
| A | desfase inicial que decae: $V = V_{AIFS} + (V_{obs,0} - V_{AIFS,0})\,e^{-L/\tau}$ | τ ajustado en entrenamiento |
| B | regresión lineal por cuantiles (mediana) | $V_{AIFS}$, $V_{AIFS,0}$, $V_{obs,0}$, plazo, interacciones con el plazo |
| C | gradient boosting (pérdida absoluta) | B + presión de AIFS, cambio de intensidad de AIFS, latitud, longitud, cuenca, traslación, día del año |

Sin predictores ambientales todavía (SST, cizalladura, humedad): ese es el paso siguiente. La validación separa
tormentas completas (GroupKFold por SID) y, aparte, temporadas completas (entrenar 2023 → probar 2024 y viceversa).
2025 no se toca.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import QuantileRegressor
from sklearn.model_selection import GroupKFold

from itchi.plotting import MapStyle, get_crameri_colormap

DATA = Path("/mnt/d/AITCHazard/tracking/match_d24")
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
FIGS = ROOT / "outputs/figures/aitchazard"
FIGS.mkdir(parents=True, exist_ok=True)
LANG = "en"
N_SPLITS = 5
TAUS = [6, 12, 24, 36, 48, 72, 120, 240, 1e9]   # h; 1e9 = constant offset
SEED = 0

STYLE = MapStyle()
CMAP = get_crameri_colormap("batlow")
plt.rcParams.update({"font.size": STYLE.label_size, "axes.linewidth": STYLE.spine_linewidth,
                     "xtick.major.width": STYLE.tick_width, "ytick.major.width": STYLE.tick_width,
                     "xtick.major.size": STYLE.tick_size, "ytick.major.size": STYLE.tick_size})
NAMES = {"raw": "AIFS raw", "pers": "Persistence", "A": "A: decaying offset", "B": "B: quantile regression",
         "C": "C: gradient boosting"}
COLORS = {k: CMAP(v) for k, v in zip(NAMES, (0.0, 0.85, 0.25, 0.5, 0.7))}

In [ ]:
long = pd.read_csv(DATA / "matches_long.csv", parse_dates=["init", "valid"])
d = long[(long["match"] == "t0") & long["ob_nature"].eq("TS") & long["fc_vmax_kt"].notna()
         & long["ob_vmax_kt"].notna() & long["fc_vmax_kt_t0"].notna() & long["ob_vmax_kt_t0"].notna()].copy()
d = d.sort_values(["init", "SID", "track", "lead_h"]).reset_index(drop=True)


def haversine_km(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, (lon1, lat1, lon2, lat2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))


# forecast translation speed (m/s) from consecutive forecast positions of the same track
g = d.groupby(["init", "SID", "track"])
dist = haversine_km(g["fc_lon"].shift(), g["fc_lat"].shift(), d["fc_lon"], d["fc_lat"])
dt = (d["lead_h"] - g["lead_h"].shift()) * 3600.0
d["trans_ms"] = dist * 1e3 / dt
d["trans_ms"] = d.groupby(["init", "SID", "track"])["trans_ms"].bfill()   # lead 0: speed of the first step
d["dfc_vmax"] = d["fc_vmax_kt"] - d["fc_vmax_kt_t0"]
d["basin_ep"] = d["basin"].eq("EP").astype(int)
d["doy"] = d["valid"].dt.dayofyear
d["year"] = d["init"].dt.year

# lead 0 is known (observed) and is not evaluated; 2025 is the locked test year and never enters the pilot
ev = d[(d["lead_h"] > 0) & d["year"].isin([2023, 2024])].reset_index(drop=True)
print(f"{len(ev)} pairs (lead > 0), {ev['SID'].nunique()} storms, years {sorted(ev['year'].unique())}")

In [ ]:
FEAT_B = ["fc_vmax_kt", "fc_vmax_kt_t0", "ob_vmax_kt_t0", "lead_h"]
FEAT_C = FEAT_B + ["fc_msl_hPa", "fc_msl_hPa_t0", "dfc_vmax", "fc_lat", "fc_lon", "basin_ep", "trans_ms", "doy"]


def design_b(df):
    X = df[FEAT_B].to_numpy(float)
    L = df["lead_h"].to_numpy(float)[:, None] / 72.0
    return np.hstack([X, X[:, :3] * L])          # intensities and their interaction with lead


def fit_predict(tr, te):
    out = {"raw": te["fc_vmax_kt"].to_numpy(), "pers": te["ob_vmax_kt_t0"].to_numpy()}
    # A: tau minimising MAE on the training storms
    off_tr = (tr["ob_vmax_kt_t0"] - tr["fc_vmax_kt_t0"]).to_numpy()
    mae = [np.abs(tr["fc_vmax_kt"] + off_tr * np.exp(-tr["lead_h"] / t) - tr["ob_vmax_kt"]).mean() for t in TAUS]
    tau = TAUS[int(np.argmin(mae))]
    out["A"] = (te["fc_vmax_kt"] + (te["ob_vmax_kt_t0"] - te["fc_vmax_kt_t0"]) * np.exp(-te["lead_h"] / tau)).to_numpy()
    # B: median linear regression
    qb = QuantileRegressor(quantile=0.5, alpha=0.0, solver="highs").fit(design_b(tr), tr["ob_vmax_kt"])
    out["B"] = qb.predict(design_b(te))
    # C: gradient boosting, absolute loss
    gb = HistGradientBoostingRegressor(loss="absolute_error", max_iter=300, learning_rate=0.05,
                                       max_leaf_nodes=15, min_samples_leaf=40, random_state=SEED)
    gb.fit(tr[FEAT_C], tr["ob_vmax_kt"])
    out["C"] = gb.predict(te[FEAT_C])
    return out, tau


def cross_validate(df, splits):
    preds = {k: np.full(len(df), np.nan) for k in NAMES}
    taus = []
    for tr_idx, te_idx in splits:
        out, tau = fit_predict(df.iloc[tr_idx], df.iloc[te_idx])
        taus.append(tau)
        for k, v in out.items():
            preds[k][te_idx] = v
    return pd.DataFrame(preds, index=df.index), taus


gkf = GroupKFold(n_splits=N_SPLITS)
p_storm, taus_storm = cross_validate(ev, gkf.split(ev, groups=ev["SID"]))
yrs = ev["year"].to_numpy()
p_year, taus_year = cross_validate(ev, [(np.where(yrs != y)[0], np.where(yrs == y)[0]) for y in sorted(set(yrs))])
print("tau (h) per fold, storms:", taus_storm, "| seasons:", taus_year)

In [ ]:
def scores(p, df):
    rows = []
    for L, idx in df.groupby("lead_h").groups.items():
        y = df.loc[idx, "ob_vmax_kt"]
        for k in NAMES:
            e = p.loc[idx, k] - y
            rows.append(dict(lead_h=L, method=k, n=len(idx), n_storms=df.loc[idx, "SID"].nunique(),
                             mae=e.abs().mean(), bias=e.mean()))
    s = pd.DataFrame(rows)
    ref = s[s["method"] == "pers"].set_index("lead_h")["mae"]
    s["skill_vs_pers"] = 1 - s["mae"] / s["lead_h"].map(ref)
    return s


s_storm, s_year = scores(p_storm, ev), scores(p_year, ev)
tab = s_storm[s_storm["lead_h"].isin([12, 24, 48, 72])].pivot(index="method", columns="lead_h", values=["mae", "bias"])
print("GroupKFold by storm"); display(tab.round(1))
tab = s_year[s_year["lead_h"].isin([12, 24, 48, 72])].pivot(index="method", columns="lead_h", values=["mae", "bias"])
print("Leave-one-season-out"); display(tab.round(1))

In [ ]:
# MAE by observed category at 48 h (storm CV): does any method reach the major hurricanes?
SS_EDGES = [0, 34, 64, 96, 400]
SS_LABELS = ["TD", "TS", "H1-H2", "H3-H5"]
z = ev[ev["lead_h"] == 48].copy()
z["cat"] = pd.cut(z["ob_vmax_kt"], SS_EDGES, right=False, labels=SS_LABELS)
cat_tab = pd.DataFrame({k: (p_storm.loc[z.index, k] - z["ob_vmax_kt"]).groupby(z["cat"], observed=False).mean()
                        for k in NAMES})
cat_tab["n"] = z.groupby("cat", observed=False).size()
print("Bias at 48 h by observed category (kt)"); display(cat_tab.round(1))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), gridspec_kw={"wspace": 0.3})
for ax, (s, title) in zip(axes[:2], ((s_storm, "CV by storm"), (s_year, "Leave-one-season-out"))):
    for k in NAMES:
        t = s[s["method"] == k]
        ax.plot(t["lead_h"], t["mae"], marker="o", ms=3, color=COLORS[k], label=NAMES[k],
                ls="--" if k in ("raw", "pers") else "-")
    ax.set(xlabel="Lead time (h)", ylabel="$V_{max}$ MAE (kt)", title=title, xticks=range(0, 73, 12))
axes[0].legend(frameon=False, fontsize=9)
ax = axes[2]
for k in NAMES:
    t = s_storm[s_storm["method"] == k]
    ax.plot(t["lead_h"], t["bias"], marker="o", ms=3, color=COLORS[k], ls="--" if k in ("raw", "pers") else "-")
ax.axhline(0, color="0.3", lw=0.8)
ax.set(xlabel="Lead time (h)", ylabel="$V_{max}$ bias (kt)", title="Bias, CV by storm", xticks=range(0, 73, 12))
for letter, a in zip("abc", axes):
    a.text(-0.02, 1.04, f"({letter})", transform=a.transAxes, fontweight="bold", ha="right")
out = FIGS / f"aifs_intensity_correction_pilot_2023_2024_{LANG}"
fig.savefig(out.with_suffix(".pdf"), bbox_inches="tight")
fig.savefig(out.with_suffix(".png"), dpi=300, bbox_inches="tight")
print("saved", out.with_suffix(".pdf"))

In [ ]:
# Which predictors carry C (permutation importance on one storm fold, MAE increase in kt)
from sklearn.inspection import permutation_importance

tr_idx, te_idx = next(GroupKFold(n_splits=N_SPLITS).split(ev, groups=ev["SID"]))
gb = HistGradientBoostingRegressor(loss="absolute_error", max_iter=300, learning_rate=0.05, max_leaf_nodes=15,
                                   min_samples_leaf=40, random_state=SEED).fit(ev.iloc[tr_idx][FEAT_C],
                                                                              ev.iloc[tr_idx]["ob_vmax_kt"])
pi = permutation_importance(gb, ev.iloc[te_idx][FEAT_C], ev.iloc[te_idx]["ob_vmax_kt"],
                            scoring="neg_mean_absolute_error", n_repeats=20, random_state=SEED)
pd.Series(pi.importances_mean, index=FEAT_C).sort_values(ascending=False).round(2)

## C2: predictores ambientales y cuantiles

`env_predictors.csv` (de `scripts/tracking/env_predictors.py`): cizalladura 850–200 hPa y humedad relativa
500/850 hPa en el anillo de 200–800 km (campos de AIFS en el tiempo válido), skt de ERA5 en t0 (sustituto de SST)
y distancia a tierra. Tres modelos en cuantiles 0.1, 0.5 y 0.9:

| Id | Modelo |
|---|---|
| B | regresión lineal por cuantiles, solo intensidades y plazo (como arriba) |
| Benv | B + predictores ambientales (lineal) |
| C2 | mediana de B + gradient boosting por cuantiles sobre el error de B, con todos los predictores |

Se evalúa el MAE de la mediana, la cobertura del intervalo 0.1–0.9 (ideal 0.8), la pérdida de cuantil y, a 48 h,
el sesgo y la fracción de H3–H5 cuya intensidad observada queda por debajo del cuantil 0.9.

In [ ]:
ENV = ["shr_ms", "rh500", "rh850", "skt_K", "dtl_km"]
env = pd.read_csv(DATA / "env_predictors.csv", parse_dates=["init"])
ev2 = ev.merge(env, on=["init", "SID", "track", "lead_h"], how="left")
assert len(ev2) == len(ev)
print("missing fraction:"); print(ev2[ENV + ["ann_frac"]].isna().mean().round(3).to_string())
display(ev2[ENV + ["ann_frac"]].describe().round(2))

In [ ]:
QS = (0.1, 0.5, 0.9)
MODELS = ("B", "Benv", "C2")
# no day of year or longitude: together with latitude they identify individual storms, and the quantile trees
# then learn storm-specific residuals in training, which shrinks the interval on unseen storms
FEAT_C2 = [f for f in FEAT_C if f not in ("doy", "fc_lon")] + ENV


def design_benv(df, skt_fill):
    X = df[ENV].copy()
    X["land"] = X["skt_K"].isna().astype(float)
    X["skt_K"] = X["skt_K"].fillna(skt_fill)
    return np.hstack([design_b(df), X.to_numpy(float)])


def qreg(q, X, y):
    return QuantileRegressor(quantile=q, alpha=0.0, solver="highs").fit(X, y)


def fit_predict_q(tr, te):
    y, out = tr["ob_vmax_kt"], {}
    fill = tr["skt_K"].median()
    for q in QS:
        out[("B", q)] = qreg(q, design_b(tr), y).predict(design_b(te))
        out[("Benv", q)] = qreg(q, design_benv(tr, fill), y).predict(design_benv(te, fill))
    b_med = qreg(0.5, design_b(tr), y)
    res = y - b_med.predict(design_b(tr))
    for q in QS:
        # strongly regularised: with ~45 independent storms, quantile trees overfit and the interval shrinks
        gb = HistGradientBoostingRegressor(loss="quantile", quantile=q, max_iter=100, learning_rate=0.05,
                                           max_depth=3, min_samples_leaf=100, l2_regularization=1.0,
                                           random_state=SEED)
        out[("C2", q)] = out[("B", 0.5)] + gb.fit(tr[FEAT_C2], res).predict(te[FEAT_C2])
    for m in MODELS:  # avoid crossing quantiles
        s = np.sort(np.vstack([out[(m, q)] for q in QS]), axis=0)
        for i, q in enumerate(QS):
            out[(m, q)] = s[i]
    return out


def cv_q(df, splits):
    preds = {(m, q): np.full(len(df), np.nan) for m in MODELS for q in QS}
    for tr_idx, te_idx in splits:
        for k, v in fit_predict_q(df.iloc[tr_idx], df.iloc[te_idx]).items():
            preds[k][te_idx] = v
    return preds


def pinball(y, p, q):
    e = y - p
    return np.mean(np.maximum(q * e, (q - 1) * e))


gq_storm = cv_q(ev2, GroupKFold(n_splits=N_SPLITS).split(ev2, groups=ev2["SID"]))
yrs2 = ev2["year"].to_numpy()
gq_year = cv_q(ev2, [(np.where(yrs2 != y)[0], np.where(yrs2 == y)[0]) for y in sorted(set(yrs2))])

In [ ]:
def q_scores(preds, df):
    y = df["ob_vmax_kt"].to_numpy()
    rows = []
    for L in sorted(df["lead_h"].unique()):
        i = (df["lead_h"] == L).to_numpy()
        for m in MODELS:
            p10, p50, p90 = (preds[(m, q)][i] for q in QS)
            rows.append(dict(lead_h=L, model=m, mae=np.mean(np.abs(p50 - y[i])), bias=np.mean(p50 - y[i]),
                             cover_80=np.mean((y[i] >= p10) & (y[i] <= p90)),
                             pinball=np.mean([pinball(y[i], preds[(m, q)][i], q) for q in QS])))
    return pd.DataFrame(rows)


qs_storm, qs_year = q_scores(gq_storm, ev2), q_scores(gq_year, ev2)
for name, s in (("CV by storm", qs_storm), ("Leave-one-season-out", qs_year)):
    print(name)
    display(s[s["lead_h"].isin([12, 24, 48, 72])].pivot(index="model", columns="lead_h",
                                                         values=["mae", "cover_80", "pinball"]).round(2))

# major hurricanes at 48 h: median bias and how often the 0.9 quantile reaches the observed intensity
z = ev2[(ev2["lead_h"] == 48) & (ev2["ob_vmax_kt"] >= 96)]
i = z.index.to_numpy()
print(f"H3-H5 at 48 h (n = {len(z)}, storms = {z['SID'].nunique()}), CV by storm")
display(pd.DataFrame({m: {"median_bias_kt": np.mean(gq_storm[(m, 0.5)][i] - z["ob_vmax_kt"]),
                          "frac_obs_below_q90": np.mean(z["ob_vmax_kt"].to_numpy() <= gq_storm[(m, 0.9)][i])}
                      for m in MODELS}).round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), gridspec_kw={"wspace": 0.3})
QCOL = {"B": CMAP(0.5), "Benv": CMAP(0.25), "C2": CMAP(0.7)}
for m in MODELS:
    t = qs_storm[qs_storm["model"] == m]
    axes[0].plot(t["lead_h"], t["mae"], marker="o", ms=3, color=QCOL[m], label=m)
    axes[1].plot(t["lead_h"], t["cover_80"], marker="o", ms=3, color=QCOL[m], label=m)
t = s_storm[s_storm["method"] == "pers"]
axes[0].plot(t["lead_h"], t["mae"], ls="--", color=COLORS["pers"], label=NAMES["pers"])
axes[1].axhline(0.8, color="0.3", lw=0.8)
axes[0].set(xlabel="Lead time (h)", ylabel="$V_{max}$ MAE of the median (kt)", xticks=range(0, 73, 12))
axes[1].set(xlabel="Lead time (h)", ylabel="Coverage of the 0.1-0.9 interval", xticks=range(0, 73, 12), ylim=(0, 1))
axes[0].legend(frameon=False, fontsize=9)
for letter, a in zip("ab", axes):
    a.text(-0.02, 1.04, f"({letter})", transform=a.transAxes, fontweight="bold", ha="right")
out = FIGS / f"aifs_intensity_correction_quantiles_2023_2024_{LANG}"
fig.savefig(out.with_suffix(".pdf"), bbox_inches="tight")
fig.savefig(out.with_suffix(".png"), dpi=300, bbox_inches="tight")
print("saved", out.with_suffix(".pdf"))